# Build a chord-preference labeling sheet

Run notebook 03 first. This notebook estimates the key, proposes a compact candidate set, and exports a CSV. The diagnostic ordering only makes labeling easier; it is not the learned recommendation.

In [ ]:
import os
from pathlib import Path
import pandas as pd

from reharmonize_models import (
    aggregate_melody_by_chord, create_candidate_sheet, estimate_key, melody_pitch_track,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
audio_setting = os.environ.get('AUDIO_PATH')
AUDIO = Path(audio_setting) if audio_setting else ROOT / 'data/song.mp3'
if not AUDIO.is_absolute():
    AUDIO = ROOT / AUDIO
DURATION = float(os.environ.get('ANALYSIS_DURATION', '30'))
VOCALS = ROOT / 'artifacts' / f'{AUDIO.stem}-vocals.wav'
CHORDS = ROOT / 'artifacts' / f'{AUDIO.stem}-smoothed.lab'
PREFERENCES = ROOT / 'data' / f'{AUDIO.stem}-preferences.csv'
AUDIO, VOCALS, CHORDS

In [ ]:
if not VOCALS.exists() or not CHORDS.exists():
    raise FileNotFoundError('Run notebooks/03_vocal_aware_analysis.ipynb first')
chords = pd.read_csv(CHORDS, sep=r'\s+', names=['start', 'end', 'chord'])
melody = melody_pitch_track(VOCALS, duration=DURATION)
segments = aggregate_melody_by_chord(chords, melody)
key = estimate_key(chords)
print({'estimated_key': key, 'segments': len(segments)})

In [ ]:
sheet = create_candidate_sheet(segments, key=key, max_candidates=8)
PREFERENCES.parent.mkdir(parents=True, exist_ok=True)
sheet.to_csv(PREFERENCES, index=False)
print(f'Saved {len(sheet)} rows to {PREFERENCES}')
sheet.head(16)

## Label the candidates

Open the exported CSV in Numbers or a spreadsheet editor. For every `segment_id`, listen to that time range and set `preferred` to `1` for exactly one candidate. Set the other candidates in that segment to `0`. Keep the CSV under `data/`; audio and private labels are ignored by Git.

In [ ]:
sheet.groupby('segment_id')[['start', 'end', 'source_chord']].first()